In [3]:
# 1. Unzip the dataset quietly
!unzip -q /content/basketball-court-detection-2.v22i.yolov8.zip -d /content/yolov8_dataset

# 2. Clear any corrupt cache files from previous runs
!rm -rf /content/yolov8_dataset/train/*.cache
!rm -rf /content/yolov8_dataset/valid/*.cache

# 3. Install required libraries
!pip install ultralytics opencv-python pyyaml

replace /content/yolov8_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [4]:
import yaml

yaml_path = '/content/yolov8_dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/yolov8_dataset'
data['train'] = 'train/images'
data['val'] = 'valid/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("Fixed data.yaml configuration!")

Fixed data.yaml configuration!


In [7]:
from ultralytics import YOLO

def train_highly_accurate_court_model():
    # 1. UPGRADE ARCHITECTURE: Switch from 'n' (Nano) to 'm' (Medium)
    # This increases parameters from ~3 million to ~25 million for vastly better spatial awareness
    model_type = 'yolov8m-pose.pt'
    print(f"Initializing high-capacity model: {model_type}")
    model = YOLO(model_type)

    print("Starting optimized training session...")
    model.train(
        data='/content/yolov8_dataset/data.yaml',
        epochs=100,
        imgsz=640,
        batch=32,
        device=0,
        workers=2,
        cache=True,
        name='court_mapping_medium_optimized',

        # 2. DISABLE AGGRESSIVE AUGMENTATIONS
        mosaic=0.0,         # Turn off image collaging
        degrees=0.0,        # Turn off random rotations
        perspective=0.0,    # Turn off perspective warping
        mixup=0.0,          # Turn off image blending

        # Keep horizontal flipping enabled (fliplr=0.5) because a court is
        # naturally symmetrical left-to-right!
    )
    print("Training Complete!")

if __name__ == "__main__":
    train_highly_accurate_court_model()

Initializing high-capacity model: yolov8m-pose.pt
Starting optimized training session...
Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov8_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-pose.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=co

KeyboardInterrupt: 

In [9]:
import cv2
import os
import numpy as np
from ultralytics import YOLO

def mark_court_keypoints_short(video_path, model_path, output_path, max_seconds=10, conf_threshold=0.5):
    """
    Runs inference using the custom court pose model, filters individual keypoints
    by confidence (> 0.5), and saves the first 10 seconds of marked footage.
    """
    if not os.path.exists(model_path):
        print(f"Error: Custom model weights not found at {model_path}")
        return

    # Load your optimized court landmark model
    print(f"Loading custom court landmark model from: {model_path}...")
    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file at {video_path}")
        return

    # Extract video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Calculate the frame limit for 10 seconds
    max_frames = fps * max_seconds

    print(f"Video Loaded: {width}x{height} | {fps} FPS")
    print(f"Processing first {max_seconds} seconds ({max_frames} frames)...")

    # Initialize the Video Writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0

    while True:
        ret, frame = cap.read()
        # Stop if video ends or if we reach the 10-second mark
        if not ret or frame_count >= max_frames:
            break

        # Run inference on the frame
        # (We set a lower overall conf to detect the court structure, but filter strictly below)
        results = model.predict(frame, conf=0.25, verbose=False)

        # Manually parse the keypoints tensor to filter out low-confidence noise
        if results[0].keypoints is not None:
            # keypoints.data[0] is an array of shape [33, 3] -> [X, Y, Confidence]
            kp_tensor = results[0].keypoints.data[0].cpu().numpy()

            for idx, kp in enumerate(kp_tensor):
                x, y, conf = kp

                # THE ROBOFLOW FILTER: Only map points if the model is highly certain
                if conf > conf_threshold:
                    # Draw a clear green dot for verified anchors
                    cv2.circle(frame, (int(x), int(y)), radius=5, color=(0, 255, 0), thickness=-1)

                    # Label the dot with its specific landmark index (0 - 32)
                    cv2.putText(frame, str(idx), (int(x) + 6, int(y) - 6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        # Save the frame to our output video
        out.write(frame)

        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Processed {frame_count}/{max_frames} frames...")

    # Clean up files
    cap.release()
    out.release()
    print(f"\nProcessing complete! Output saved to: {output_path}")

# --- Run configuration ---
if __name__ == "__main__":
    # Path to your uploaded clip
    INPUT_VIDEO = "/stephcurryvideo.mp4"

    # Points to the folder generated by your optimized training script
    TRAINED_MODEL = "/content/runs/pose/court_mapping_medium_optimized-3/weights/best.pt"

    OUTPUT_VIDEO = "/content/StephVideo_court_landmarks.mp4"

    # Run the function
    mark_court_keypoints_short(
        video_path=INPUT_VIDEO,
        model_path=TRAINED_MODEL,
        output_path=OUTPUT_VIDEO,
        max_seconds=10,
        conf_threshold=0.5 # Keeps only points with >50% confidence as shown in the tutorial
    )

Loading custom court landmark model from: /content/runs/pose/court_mapping_medium_optimized-3/weights/best.pt...
Video Loaded: 720x1280 | 29 FPS
Processing first 10 seconds (290 frames)...


IndexError: index 0 is out of bounds for dimension 0 with size 0